# Yelp Universal Business Dashboard

**Course:** MSBA 503 – Analytics Programming II  
**Team:** Alex, Eddie, Lily  
**Project:** Yelp-Based Universal Business Analytics Dashboard  

This notebook implements a modular, scalable `yelp_dashboard()` master function that:

- Accepts user/industry filters (Hair, Mexican Food, etc.)
- Loads Yelp review data and Business Aggregated data
- Performs descriptive, diagnostic, predictive, and prescriptive analysis
- Launches an interactive Streamlit dashboard for business users

> **Design constraints:**  
> - Do **not** rename any existing columns or files.  
> - Drop overly granular `_pct` columns (e.g., `adv_pct`, `noun_pct`, `adj_pct`, `verb_pct`) **only during data loading**, not in the source files.  
> - Final code should end with a callable `yelp_dashboard()` master function.

In [ ]:
# Install Streamlit if not already installed
!pip install streamlit --quiet

In [ ]:
#2. Imports & File Paths

# Core libraries
import polars as pl
import pandas as pd
import numpy as np

# Dashboard / visualization
import streamlit as st
import altair as alt

# Modeling & time series
from statsmodels.tsa.arima.model import ARIMA

# Utilities
import random
import time
from datetime import datetime, timedelta
import re
import warnings
import os

# Suppress common warnings during development
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# File paths (match your actual folder)
# ---------------------------------------------------------------------
SAMPLE_PATH_HAIR = "hair_sample_15k.parquet"
SAMPLE_PATH_MEXICAN = "chipotle_sample_15k.parquet"
BUSINESS_AGGREGATED_PATH = "business_aggregated_sample.csv"

# NOTE:
# - Do NOT rename or move these files in the final project.
# - The actual .parquet and .csv files should already be in the /Datasets folder
#   relative to this notebook/script.

In [41]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Helper & Utility Functions (Scaffold)
Below we define reusable components for:

- Data loading & preprocessing  
- Filtering based on user selections  
- Shared utilities for analysis functions  

We will gradually fill these functions in as we refactor Lily's prototype code.

In [ ]:
#3. Helper & Utility Functions (Scaffold)

def load_and_preprocess_data(industry_choice: str):
    """
    Load Yelp review data (Hair or Mexican Food) and Business Aggregated data.

    - Uses Polars for performance.
    - Drops overly granular *_pct columns (adv_pct, noun_pct, adj_pct, verb_pct)
      ONLY in-memory, not from the source files.
    - Joins reviews with business-level aggregated features on `business_id`.

    Returns
    -------
    df_joined : pl.DataFrame
        Joined review + business features for the selected industry.
    business_lookup : pl.DataFrame
        Unique (business_id, name, city, state, ...) rows for UI filtering.
    """
    st.info(f"Loading Yelp Data for {industry_choice}...")

    # 1. Choose the correct parquet path based on industry
    path = SAMPLE_PATH_MEXICAN if industry_choice == "Mexican Food" else SAMPLE_PATH_HAIR

    if not os.path.exists(path):
        st.error(f"Review Parquet file not found at {path}.")
        return None, None

    # 2. Load review data (Parquet)
    reviews_df = pl.read_parquet(path)

    # 3. Load business aggregated data (CSV)
    st.info("Loading Business Aggregated Data...")
    try:
        business_df = pl.read_csv(BUSINESS_AGGREGATED_PATH)
    except pl.ComputeError:
        st.error(f"Business Aggregated CSV not found at {BUSINESS_AGGREGATED_PATH}.")
        return None, None

    # 4. Drop overly granular *_pct columns if present
    cols_to_drop = ['adv_pct', 'noun_pct', 'adj_pct', 'verb_pct']
    existing_to_drop = [c for c in cols_to_drop if c in reviews_df.columns]
    if existing_to_drop:
        reviews_df = reviews_df.drop(existing_to_drop)

    # 5. Join on business_id
    st.info("Now joining industry data on 'business_id'...")
    df_joined = reviews_df.join(business_df, on='business_id', how='left')

    # 6. Mock SEC / financial statistics (placeholder logic)
    st.info("Finding Financial Statistics (MOCK)...")
    if 'avg_star' in df_joined.columns and 'stars' in df_joined.columns:
        df_joined = df_joined.with_columns(
            (
                pl.col('avg_star') * 10000 +
                pl.col('stars') * 5000 +
                pl.lit(random.randint(50_000, 200_000))
            ).alias('yearly_revenue')
        )
    else:
        # If these columns don't exist, still create a placeholder yearly_revenue
        df_joined = df_joined.with_columns(
            pl.lit(random.randint(50_000, 200_000)).alias('yearly_revenue')
        )

        # 7. Build a lookup table for business-level info
    base_cols = ['business_id', 'name', 'city', 'state']

    # Try to bring through business-level star metrics from the aggregated data
    star_cols = [
        'weighted_star_avg',
        'business_weighted_star_avg',
        'business_star_avg',
        'stars'
    ]
    lookup_cols = [c for c in base_cols + star_cols if c in df_joined.columns]

    business_lookup = df_joined.select(lookup_cols).unique()

    return df_joined, business_lookup


def apply_user_filters(df_full: pl.DataFrame,
                       business_lookup: pl.DataFrame,
                       business_id_filter=None,
                       business_name=None,
                       state_filter=None,
                       city_filter=None):
    """
    Apply user-specified filters in the correct priority order:

    Priority
    --------
    1. Exact business_id (highest priority)
    2. Business name
    3. City (within a state)
    4. State
    5. Otherwise: industry-wide

    Returns
    -------
    df_analysis : pl.DataFrame
        Filtered dataset ready for analysis.
    subset_label : str
        Human-readable label describing the subset (for dashboard display).
    """
    df_filtered = df_full.lazy()
    subset_label = "Industry-wide"

    # 1. Business ID filter
    if business_id_filter:
        df_filtered = df_filtered.filter(pl.col('business_id') == business_id_filter)
        subset_label = f"Business ID: {business_id_filter}"

    # 2. Business name filter
    elif business_name and business_name != "Industry-Level":
        if 'name' in business_lookup.columns:
            biz_ids = (
                business_lookup
                .filter(pl.col('name') == business_name)
                .select('business_id')
                .to_series()
                .to_list()
            )
            df_filtered = df_filtered.filter(pl.col('business_id').is_in(biz_ids))
            subset_label = f"Business: {business_name}"

    # 3. City filter
    elif city_filter and 'city' in df_full.columns:
        df_filtered = df_filtered.filter(pl.col('city') == city_filter)
        if state_filter and 'state' in df_full.columns:
            df_filtered = df_filtered.filter(pl.col('state') == state_filter)
            subset_label = f"City: {city_filter}, {state_filter}"
        else:
            subset_label = f"City: {city_filter}"

    # 4. State filter
    elif state_filter and 'state' in df_full.columns:
        df_filtered = df_filtered.filter(pl.col('state') == state_filter)
        subset_label = f"State: {state_filter}"

    # 5. Otherwise: industry-wide
    else:
        subset_label = "Industry-wide"

    df_analysis = df_filtered.collect()
    return df_analysis, subset_label

def descriptive_analysis(df: pl.DataFrame,
                         business_lookup: pl.DataFrame):
    """
    Descriptive analysis & visuals.

    Outputs
    -------
    line_chart : alt.Chart
        Star ratings over time (Actual vs Time-decay weighted).
    pie_chart : alt.Chart or None
        Distribution of dominant emotion/sentiment (if column exists).
    top_5_best : pl.DataFrame
        Top 5 locations by weighted star rating.
    top_5_worst : pl.DataFrame
        Bottom 5 locations by weighted star rating.
    """

    # --- 1. Star Ratings over Time (Actual vs Weighted) ---

    if "review_date" not in df.columns:
        raise ValueError("Column 'review_date' not found in dataframe.")

    # Try to parse review_date to monthly datetime
    try:
        df_ts = df.with_columns(
            pl.col("review_date")
            .str.strptime(pl.Date, strict=False)
            .cast(pl.Datetime)
            .dt.truncate("1mo")
        )
    except Exception:
        # Fallback: assume it's already date/datetime-like
        df_ts = df.with_columns(
            pl.col("review_date")
            .cast(pl.Datetime)
            .dt.truncate("1mo")
        )

        # Choose a weighted-star column if available, else fall back to raw stars
    if "weighted_star_avg" in df_ts.columns:
        # This should be on the same 1–5 scale as stars
        weight_col = "weighted_star_avg"
    elif "weighted_star" in df_ts.columns:
        # This is likely a raw weight (0–1). We *don't* want that as a rating,
        # so only use it if it's clearly on a 1–5 scale.
        mean_val = df_ts.select(pl.col("weighted_star").mean()).item()
        if mean_val is not None and mean_val > 1.0:
            weight_col = "weighted_star"
        else:
            weight_col = "stars"
    else:
        weight_col = "stars"

        # Figure out if the weighted column is on a 0–1 scale; if so, rescale to 0–5
    max_weight = df_ts.select(pl.col(weight_col).max()).item()
    if max_weight is not None and max_weight <= 1.5:
        scale_factor = 5.0   # bring 0–1 (or 0–0.2) up to 0–5
    else:
        scale_factor = 1.0   # already on 1–5-ish scale

        # Compute means to align scales
    stars_mean = df_ts.select(pl.col("stars").mean()).item()
    weighted_mean = df_ts.select(pl.col(weight_col).mean()).item()

    # If weighted_mean is tiny (0–0.5), rescale so its mean matches stars_mean
    if weighted_mean is not None and weighted_mean > 0:
        scale_factor = stars_mean / weighted_mean
    else:
        scale_factor = 1.0

    df_ts_agg = (
        df_ts
        .group_by("review_date")
        .agg(
            pl.col("stars").mean().alias("Actual_Stars"),
            (pl.col(weight_col) * scale_factor).mean().alias("Weighted_Stars"),
        )
        .sort("review_date")
    )

    df_chart = (
        df_ts_agg
        .melt(
            id_vars="review_date",
            value_vars=["Actual_Stars", "Weighted_Stars"],
            variable_name="Rating_Type",
            value_name="Rating",
        )
        .to_pandas()
    )

    line_chart = (
    alt.Chart(df_chart)
    .mark_line(point=False, strokeWidth=2)
    .encode(
        x=alt.X(
            "review_date:T",
            title="Date",
            axis=alt.Axis(format="%Y", labelAngle=-45)
        ),
        y=alt.Y(
            "Rating:Q",
            title="Average Star Rating",
            scale=alt.Scale(domain=[0,5])
        ),
        color=alt.Color(
            "Rating_Type:N",
            title="Rating Type",
            scale=alt.Scale(scheme="dark2")
        ),
        tooltip=[
            alt.Tooltip("review_date:T", title="Date"),
            alt.Tooltip("Rating_Type:N", title="Type"),
            alt.Tooltip("Rating:Q", title="Avg Rating", format=".2f"),
        ],
    )
    .properties(
        title="Star Ratings (Actual vs Time-Decay Weighted) Over Time",
        width=700,
        height=400
    )
    .interactive()  # enables zoom/pan
)

    # --- 2. Distribution of Dominant Emotion (Pie Chart) ---

    pie_chart = None
    emotion_col = None

    if "dominant_emotion" in df.columns:
        emotion_col = "dominant_emotion"
        df_pie = (
            df
            .group_by(emotion_col)
            .agg(pl.count().alias("count"))
            .sort("count", descending=True)
        )
    elif "primary_emotion_mode" in df.columns:
        emotion_col = "primary_emotion_mode"
        df_pie = (
            df
            .group_by(emotion_col)
            .agg(pl.count().alias("count"))
            .sort("count", descending=True)
        )
    elif "primary_emotion_mode" in business_lookup.columns:
        emotion_col = "primary_emotion_mode"
        df_pie = (
            business_lookup
            .group_by(emotion_col)
            .agg(pl.count().alias("count"))
            .sort("count", descending=True)
        )

    if emotion_col is not None:
        pie_chart = (
            alt.Chart(df_pie.to_pandas())
            .mark_arc(outerRadius=120)
            .encode(
                theta=alt.Theta(field="count", type="quantitative", stack=True),
                color=alt.Color(field=emotion_col, type="nominal",
                                title="Dominant Emotion"),
                order=alt.Order(field="count", sort="descending"),
                tooltip=[emotion_col, "count"],
            )
            .properties(title="Distribution of Dominant Emotions")
        )

        # --- 3. Top 5 Best / Worst Locations ---

    # Compute business-level average stars directly from the review data
    if "business_id" not in df.columns or "stars" not in df.columns:
        raise ValueError("Columns 'business_id' and 'stars' are required for top 5 tables.")

    business_scores = (
        df
        .group_by("business_id")
        .agg([
            pl.col("stars").mean().alias("avg_stars"),
            pl.count().alias("review_count"),
        ])
    )

    # Bring in name / city / state from the lookup table
    business_scores = business_scores.join(
        business_lookup,
        on="business_id",
        how="left"
    )

    # Top 5 best (highest avg_stars)
    top_5_best = (
        business_scores
        .sort("avg_stars", descending=True)
        .head(5)
        .select(["business_id", "name", "city", "state", "avg_stars", "review_count"])
    )

    # Top 5 worst (lowest avg_stars)
    top_5_worst = (
        business_scores
        .sort("avg_stars", descending=False)
        .head(5)
        .select(["business_id", "name", "city", "state", "avg_stars", "review_count"])
    )

    return line_chart, pie_chart, top_5_best, top_5_worst

## Descriptive Analysis

In [ ]:
def descriptive_analysis(df: pl.DataFrame,
                         business_lookup: pl.DataFrame):
    """
    Descriptive analysis & visuals.

    Outputs
    -------
    line_chart : alt.Chart
        Star ratings over time (Actual vs Time-decay weighted).
    pie_chart : alt.Chart or None
        Distribution of dominant emotion/sentiment (if column exists).
    top_5_best : pl.DataFrame
        Top 5 locations by average star rating.
    top_5_worst : pl.DataFrame
        Bottom 5 locations by average star rating.
    """

    # -----------------------------
    # 1. Star Ratings Over Time
    # -----------------------------
    if "review_date" not in df.columns:
        raise ValueError("Column 'review_date' not found in dataframe.")

    # Parse dates to monthly buckets
    try:
        df_ts = df.with_columns(
            pl.col("review_date")
            .str.strptime(pl.Date, strict=False)
            .cast(pl.Datetime)
            .dt.truncate("1mo")
        )
    except Exception:
        df_ts = df.with_columns(
            pl.col("review_date")
            .cast(pl.Datetime)
            .dt.truncate("1mo")
        )

    # Choose a weighted-star column
    if "weighted_star_avg" in df_ts.columns:
        weight_col = "weighted_star_avg"
    elif "weighted_star" in df_ts.columns:
        weight_col = "weighted_star"
    else:
        weight_col = "stars"

    # Scale weighted so its mean is comparable to stars
    stars_mean = df_ts.select(pl.col("stars").mean()).item()
    weighted_mean = df_ts.select(pl.col(weight_col).mean()).item()
    if weighted_mean is not None and weighted_mean > 0:
        scale_factor = stars_mean / weighted_mean
    else:
        scale_factor = 1.0

    df_ts_agg = (
        df_ts
        .group_by("review_date")
        .agg(
            pl.col("stars").mean().alias("Actual_Stars"),
            (pl.col(weight_col) * scale_factor).mean().alias("Weighted_Stars"),
        )
        .sort("review_date")
    )

    df_chart = (
        df_ts_agg
        .melt(
            id_vars="review_date",
            value_vars=["Actual_Stars", "Weighted_Stars"],
            variable_name="Rating_Type",
            value_name="Rating",
        )
        .to_pandas()
    )

    line_chart = (
        alt.Chart(df_chart)
        .mark_line(point=False, strokeWidth=2)
        .encode(
            x=alt.X(
                "review_date:T",
                title="Date",
                axis=alt.Axis(format="%Y", labelAngle=-45)
            ),
            y=alt.Y(
                "Rating:Q",
                title="Average Star Rating",
                scale=alt.Scale(domain=[0, 5])
            ),
            color=alt.Color(
                "Rating_Type:N",
                title="Rating Type"
            ),
            tooltip=[
                alt.Tooltip("review_date:T", title="Date"),
                alt.Tooltip("Rating_Type:N", title="Type"),
                alt.Tooltip("Rating:Q", title="Avg Rating", format=".2f"),
            ],
        )
        .properties(
            title="Star Ratings (Actual vs Time-Decay Weighted) Over Time",
            width=700,
            height=400
        )
        .interactive()
    )

    # -----------------------------
    # 2. Emotion Distribution
    # -----------------------------
    pie_chart = None
    emotion_col = None

    if "dominant_emotion" in df.columns:
        emotion_col = "dominant_emotion"
        df_pie = (
            df
            .group_by(emotion_col)
            .agg(pl.count().alias("count"))
            .sort("count", descending=True)
        )
    elif "primary_emotion_mode" in df.columns:
        emotion_col = "primary_emotion_mode"
        df_pie = (
            df
            .group_by(emotion_col)
            .agg(pl.count().alias("count"))
            .sort("count", descending=True)
        )
    elif "primary_emotion_mode" in business_lookup.columns:
        emotion_col = "primary_emotion_mode"
        df_pie = (
            business_lookup
            .group_by(emotion_col)
            .agg(pl.count().alias("count"))
            .sort("count", descending=True)
        )
    else:
        df_pie = None

    if emotion_col is not None and df_pie is not None:
        pie_chart = (
            alt.Chart(df_pie.to_pandas())
            .mark_arc(outerRadius=120)
            .encode(
                theta=alt.Theta(field="count", type="quantitative", stack=True),
                color=alt.Color(field=emotion_col, type="nominal",
                                title="Dominant Emotion"),
                order=alt.Order(field="count", sort="descending"),
                tooltip=[emotion_col, "count"],
            )
            .properties(title="Distribution of Dominant Emotions")
        )

    # -----------------------------
    # 3. Top 5 Best / Worst Locations
    # -----------------------------
    if "business_id" not in df.columns or "stars" not in df.columns:
        raise ValueError("Columns 'business_id' and 'stars' are required for top 5 tables.")

    business_scores = (
        df
        .group_by("business_id")
        .agg([
            pl.col("stars").mean().alias("avg_stars"),
            pl.count().alias("review_count"),
        ])
    )

    # Join on lookup to get name/city/state
    business_scores = business_scores.join(
        business_lookup,
        on="business_id",
        how="left"
    )

    top_5_best = (
        business_scores
        .sort("avg_stars", descending=True)
        .head(5)
        .select(["business_id", "name", "city", "state", "avg_stars", "review_count"])
    )

    top_5_worst = (
        business_scores
        .sort("avg_stars", descending=False)
        .head(5)
        .select(["business_id", "name", "city", "state", "avg_stars", "review_count"])
    )

    return line_chart, pie_chart, top_5_best, top_5_worst

## Predictive Analysis

In [ ]:
def predictive_analysis(df: pl.DataFrame, periods: int = 6):
    """
    Predict future average star ratings using ARIMA.

    Parameters
    ----------
    df : pl.DataFrame
        Filtered review dataset.
    periods : int
        Number of months to forecast.

    Returns
    -------
    forecast_chart : alt.Chart or None
        Chart with historical + forecasted ratings.
    predicted_star : float
        Star prediction for the final forecasted month.
    """

    # ---------------------------
    # 1. Ensure review_date exists
    # ---------------------------
    if "review_date" not in df.columns or "stars" not in df.columns:
        raise ValueError("DataFrame must contain 'review_date' and 'stars' columns.")

    # ---------------------------
    # 2. Parse review_date → month
    # ---------------------------
    try:
        df_ts = df.with_columns(
            pl.col("review_date")
            .str.strptime(pl.Date, strict=False)
            .cast(pl.Datetime)
            .dt.truncate("1mo")
        )
    except Exception:
        df_ts = df.with_columns(
            pl.col("review_date")
            .cast(pl.Datetime)
            .dt.truncate("1mo")
        )

    # ---------------------------
    # 3. Build monthly star series
    # ---------------------------
    ts_df = (
        df_ts
        .group_by("review_date")
        .agg(pl.col("stars").mean().alias("mean_star"))
        .sort("review_date")
        .to_pandas()
        .set_index("review_date")
    )

    # Resample monthly (ensure continuous)
    ts_df = ts_df.resample("M").ffill()

    # If dataset too small → fallback
    if len(ts_df) < 12:
        last_val = float(ts_df["mean_star"].iloc[-1])
        return None, last_val

    # ---------------------------
    # 4. Fit ARIMA Model
    # ---------------------------
    try:
        model = ARIMA(ts_df["mean_star"], order=(5, 1, 0))
        model_fit = model.fit()

        # Forecast next N months
        forecast = model_fit.predict(
            start=len(ts_df),
            end=len(ts_df) + periods - 1
        )

        # Build forecast index
        forecast_index = [
            ts_df.index[-1] + pd.DateOffset(months=i+1)
            for i in range(periods)
        ]

        forecast_df = pd.DataFrame({
            "review_date": forecast_index,
            "mean_star": forecast.values,
            "type": "Forecast"
        })

        # Historical df
        hist_df = ts_df.reset_index()
        hist_df["type"] = "Historical"

        # Combine for plotting
        combined_df = pd.concat([hist_df, forecast_df], ignore_index=True)

                # ---------------------------
        # 5. Build Altair Forecast Chart
        # ---------------------------

        # Separate historical and forecast for clean layering
        hist_pd = hist_df.copy()
        fc_pd = forecast_df.copy()

        chart_hist = (
            alt.Chart(hist_pd)
            .mark_line(color="#f28e2b", strokeWidth=2)  # Orange
            .encode(
                x=alt.X("review_date:T", title="Date"),
                y=alt.Y("mean_star:Q", title="Average Star Rating",
                        scale=alt.Scale(domain=[0, 5])),
                tooltip=[
                    alt.Tooltip("review_date:T", title="Date"),
                    alt.Tooltip("mean_star:Q", title="Avg Stars", format=".2f"),
                ],
            )
        )

        chart_fc = (
            alt.Chart(fc_pd)
            .mark_line(color="#4e79a7", strokeWidth=2)  # Blue
            .encode(
                x="review_date:T",
                y="mean_star:Q",
                tooltip=[
                    alt.Tooltip("review_date:T", title="Date"),
                    alt.Tooltip("mean_star:Q", title="Forecast", format=".2f"),
                ],
            )
        )

        forecast_chart = (
            (chart_hist + chart_fc)
            .properties(
                title=f"Star Rating Forecast ({periods} Months)",
                width=700,
                height=400,
            )
        )

        predicted_star = float(forecast.values[-1])

        return forecast_chart, predicted_star

    except Exception as e:
        print("ARIMA failed:", e)
        last_val = float(ts_df["mean_star"].iloc[-1])
        return None, last_val

# Analytics Functions (Diagnostic & Prescriptive – Eddie’s Section)


In [ ]:
# -------------------------------------------------------------------
# 4. Analytics Functions (Diagnostic & Prescriptive – Eddie’s Section)
# -------------------------------------------------------------------
# DIAGNOSTIC:
def diagnostic_analysis(
    df: pl.DataFrame,
    business_lookup: pl.DataFrame,
    top_n: int = 20
):
    """
    Diagnostic analysis (Eddie's section).

    Identifies:
    - Pain points using keyword frequency in low-star reviews.
    - Emotion intensity summaries (anger, fear, joy, etc.) when available.
    - Linguistic correlations with star ratings.
    - Business-level –at risk’ flags based on recent review declines.

    Returns
    -------
    dict
        {
          'pain_point_chart': alt.Chart or None,
          'pain_points_df': pandas.DataFrame,
          'emotion_summary': pandas.DataFrame or None,
          'linguistic_correlations': pandas.DataFrame,
          'at_risk_businesses': pandas.DataFrame
        }
    """
    import pandas as pd
    import numpy as np
    import altair as alt
    import re

    STOPWORDS = {
        'the','and','for','that','this','with','you','have','are','but','not','was','were',
        'they','from','your','their','has','had','our','all','too','just','then','what',
        'when','there','which','will','been','into','out','about','like','theyre','dont','cant'
    }

    def _tokenize(text: str):
        if not isinstance(text, str):
            return []
        words = re.findall(r"\b[a-zA-Z]{3,}\b", text.lower())
        return [w for w in words if w not in STOPWORDS]

    text_col = None
    for c in ["stripped_review", "text", "raw_review"]:
        if c in df.columns:
            text_col = c
            break

    pain_points_df = pd.DataFrame(columns=["Word", "Frequency"])
    pain_point_chart = None

    if "stars" in df.columns and text_col is not None:
        low_df = df.filter(pl.col("stars") < 3.0)
        if len(low_df) > 0:
            texts = low_df.select(text_col).to_series().to_list()
            counts = {}
            for t in texts:
                for w in _tokenize(t):
                    counts[w] = counts.get(w, 0) + 1
            if counts:
                top_items = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
                pain_points_df = pd.DataFrame(top_items, columns=["Word", "Frequency"])
                pain_point_chart = (
                    alt.Chart(pain_points_df)
                    .mark_bar()
                    .encode(
                        x=alt.X("Frequency:Q", title="Frequency in Low-Rated Reviews"),
                        y=alt.Y("Word:N", sort="-x", title="Pain Point"),
                        tooltip=["Word", "Frequency"]
                    )
                    .properties(title="Top Pain Points", width=600, height=400)
                )

    emotion_cols = [
        c for c in df.columns
        if c.endswith("_int_avg") or (
            c.endswith("_pct") and any(k in c for k in ["anger","fear","joy","sad","disgust"])
        )
    ]
    emotion_summary = None

    if emotion_cols:
        averages = {}
        for c in emotion_cols:
            try:
                averages[c] = float(df.select(pl.col(c).mean()).item() or 0.0)
            except:
                averages[c] = np.nan
        if "stars" in df.columns:
            try:
                averages["avg_stars"] = float(df.select(pl.col("stars").mean()).item())
            except:
                averages["avg_stars"] = np.nan
        emotion_summary = pd.DataFrame([averages])
    elif "dominant_emotion" in df.columns:
        emotion_summary = (
            df.group_by("dominant_emotion")
              .agg(pl.count())
              .sort("count", descending=True)
              .rename({"count": "review_count"})
              .to_pandas()
        )

    ling_cols = [
        c for c in df.columns
        if c in ["noun_pct","adj_pct","verb_pct","adv_pct","noun_pct_avg","adj_pct_avg"]
    ]
    ling_corr_df = None

    if "stars" in df.columns and ling_cols:
        sub = df.select(["stars"] + ling_cols).to_pandas().dropna()
        if len(sub) > 0:
            corrs = sub.corr(method="pearson")["stars"].drop("stars")
            ling_corr_df = (
                corrs.reset_index()
                     .rename(columns={"index": "feature", "stars": "pearson_corr_with_stars"})
            )

    at_risk_df = pd.DataFrame()

    if all(x in df.columns for x in ["business_id", "review_date", "stars"]):
        try:
            df_dt = df.with_columns(
                pl.col("review_date")
                  .str.strptime(pl.Date, strict=False)
                  .cast(pl.Datetime)
                  .dt.truncate("1mo")
            )
        except:
            df_dt = df.with_columns(
                pl.col("review_date").cast(pl.Datetime).dt.truncate("1mo")
            )

        now = df_dt.select(pl.col("review_date").max()).item()
        now = pd.to_datetime(now) if now else pd.Timestamp.now()

        six_months_ago = now - pd.DateOffset(months=6)
        twelve_months_ago = now - pd.DateOffset(months=12)

        recent = df_dt.filter(pl.col("review_date") >= six_months_ago)
        prior  = df_dt.filter(
            (pl.col("review_date") >= twelve_months_ago) &
            (pl.col("review_date") < six_months_ago)
        )

        recent_stats = recent.group_by("business_id").agg([
            pl.col("stars").mean().alias("recent_mean_star"),
            (pl.col("stars") < 3.0).sum().alias("recent_low_count"),
            pl.count().alias("recent_review_count")
        ])
        prior_stats = prior.group_by("business_id").agg([
            pl.col("stars").mean().alias("prior_mean_star"),
            (pl.col("stars") < 3.0).sum().alias("prior_low_count"),
            pl.count().alias("prior_review_count")
        ])

        risk = recent_stats.join(prior_stats, on="business_id", how="left").to_pandas().fillna(0)
        risk["decline"] = risk["prior_mean_star"] - risk["recent_mean_star"]
        risk["recent_low_pct"] = np.where(
            risk["recent_review_count"] > 0,
            risk["recent_low_count"] / risk["recent_review_count"],
            0
        )

        def _flag(row):
            return (row["decline"] > 0.3) or (
                row["recent_low_pct"] > 0.25 and row["recent_review_count"] >= 10
            )

        risk["at_risk_flag"] = risk.apply(_flag, axis=1)

        if "business_id" in business_lookup.columns:
            risk = risk.merge(business_lookup.to_pandas(), on="business_id", how="left")

        at_risk_df = risk.sort_values(["at_risk_flag", "decline"], ascending=[False, False])

    return {
        "pain_point_chart": pain_point_chart,
        "pain_points_df": pain_points_df,
        "emotion_summary": emotion_summary,
        "linguistic_correlations": ling_corr_df if ling_corr_df is not None else pd.DataFrame(),
        "at_risk_businesses": at_risk_df
    }

In [ ]:
# PRESCRIPTIVE:
def prescriptive_analysis(
    df: pl.DataFrame,
    business_lookup: pl.DataFrame = None
):
    """
    Prescriptive analysis (Eddie).

    Uses:
    - Emotion intensity (anger, fear, joy)
    - Recent declines in stars
    - Business-level risk detection
    to generate recommended actions and an improvement metric.

    Returns
    -------
    dict
        {
          'recommendation' : str,
          'action_metric' : float or None,
          'recommended_actions' : list[str],
          'business_flags' : pandas.DataFrame
        }
    """
    import pandas as pd

    avg_stars = (
        float(df.select(pl.col("stars").mean()).item())
        if "stars" in df.columns else None
    )

    anger_col = next((c for c in df.columns if c.startswith("anger") and c.endswith("_int_avg")), None)
    fear_col  = next((c for c in df.columns if c.startswith("fear")  and c.endswith("_int_avg")), None)
    joy_col   = next((c for c in df.columns if c.startswith("joy")   and c.endswith("_int_avg")), None)

    avg_anger = float(df.select(pl.col(anger_col).mean()).item()) if anger_col else 0.0
    avg_fear  = float(df.select(pl.col(fear_col).mean()).item())  if fear_col else 0.0
    avg_joy   = float(df.select(pl.col(joy_col).mean()).item())   if joy_col else 0.0

    recommendation = "Maintain current efforts and monitor sentiment trends."
    recommended_actions = []
    action_metric = avg_stars

    if avg_stars is not None and avg_stars < 4.0 and (avg_anger > 0.015 or avg_fear > 0.015):
        recommendation = (
            "🚨 HIGH PRIORITY: Elevated negative emotions detected alongside lower star ratings. "
            "Implement immediate service quality improvements and targeted outreach."
        )
        recommended_actions = [
            "Launch complaint-resolution training for front-line staff",
            "Reach out to recent 1–2 star reviewers with recovery incentives",
            "Audit operations for recurring service failures"
        ]
        action_metric = min(5.0, avg_stars + (avg_anger + avg_fear) * 10)

    else:
        diag = diagnostic_analysis(df, business_lookup)
        at_risk = diag.get("at_risk_businesses", pd.DataFrame())

        if isinstance(at_risk, pd.DataFrame) and not at_risk.empty:
            top = at_risk.iloc[0]
            decline = top.get("decline", 0)

            recommendation = (
                f"⚠️ MODERATE PRIORITY: Business '{top.get('name', top.get('business_id'))}' "
                f"shows a recent decline of {decline:.2f} stars. Investigate staffing, product, or operational issues."
            )
            recommended_actions = [
                "Conduct operational root-cause analysis",
                "Survey recent customers for qualitative insights",
                "Stabilize service delivery with short-term interventions"
            ]
            action_metric = float(decline)

        else:
            recommendation = (
                "✅ LOW PRIORITY: No negative trends detected. "
                "Focus on marketing, loyalty initiatives, and incremental improvements."
            )
            recommended_actions = [
                "Monitor sentiment monthly",
                "Expand loyalty/rewards programs",
                "A/B test small service improvements"
            ]
            action_metric = avg_stars

    return {
        "recommendation": recommendation,
        "action_metric": action_metric,
        "recommended_actions": recommended_actions,
        "business_flags": diag.get("at_risk_businesses", pd.DataFrame())
    }


In [ ]:
#5. Master Function Scaffold

def yelp_dashboard():
    """
    Master function for the Yelp Universal Business Dashboard.

    High-level flow:
    1. Configure Streamlit page and sidebar.
    2. Capture user input (industry, business, state, city, business_id).
    3. Load & preprocess data for the chosen industry.
    4. Apply user filters to create the analysis subset.
    5. Run analytics:
        - Descriptive (Alex)
        - Diagnostic (Eddie)
        - Predictive (Alex)
        - Prescriptive (Eddie)
    6. Build and launch the Streamlit dashboard layout.
    """
    # ------------------------------------------------------------------
    # 1. Streamlit Page Setup & User Input (will use helper functions)
    # ------------------------------------------------------------------
    # TODO (Alex): later, move Lily's sidebar + input widgets in here
    # or into a separate `user_input_section` helper.
    pass

    # ------------------------------------------------------------------
    # 2. Data Loading & Preprocessing
    # ------------------------------------------------------------------
    # TODO (Alex): call load_and_preprocess_data(...) here.
    pass

    # ------------------------------------------------------------------
    # 3. Apply User Filters
    # ------------------------------------------------------------------
    # TODO (Alex): call apply_user_filters(...) here.
    pass

    # ------------------------------------------------------------------
    # 4. Run Analytics
    # ------------------------------------------------------------------
    # TODO: call descriptive_analysis, diagnostic_analysis,
    #       predictive_analysis, prescriptive_analysis.
    pass

    # ------------------------------------------------------------------
    # 5. Dashboard Layout & Display
    # ------------------------------------------------------------------
    # TODO (Alex): recreate & improve Lily's Streamlit layout using:
    #   - st.metric
    #   - st.altair_chart
    #   - st.table
    #   - st.success / st.warning, etc.
    pass

In [ ]:
# 6. Entry Point / Function Call

# NOTE:
# - When running as a Streamlit app, you'll typically do:
#     streamlit run yelp_universal_dashboard.py
#   and Streamlit will execute this file.
# - For the final submission, your professor may want to see that
#   `yelp_dashboard()` is callable from the bottom of the script.

if __name__ == "__main__":
    # In a pure Python script, this would launch the dashboard.
    # In Jupyter, this line will not create a visible Streamlit app,
    # but it's included to mirror production usage.
    # Comment this out while you are still scaffolding / testing.
    # yelp_dashboard()
    pass

## CODE TESTS

In [ ]:
# TEST 1: Load updated data for both industries

#Hair
df_hair, lookup_hair = load_and_preprocess_data("Hair")
print("Hair data shape:", df_hair.shape if df_hair is not None else "No data loaded")
print("Hair lookup preview:")
print(lookup_hair.head() if lookup_hair is not None else "No lookup loaded")

print("\n" + "-"*60 + "\n")

#Mexican Food
df_mex, lookup_mex = load_and_preprocess_data("Mexican Food")
print("Mexican data shape:", df_mex.shape if df_mex is not None else "No data loaded")
print("Mexican lookup preview:")
print(lookup_mex.head() if lookup_mex is not None else "No lookup loaded")

2026-02-23 19:21:50.042 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 19:21:50.043 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 19:21:50.044 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 19:21:50.045 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 19:21:50.045 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 19:21:50.046 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 19:21:50.047 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 19:21:50.048 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

Hair data shape: No data loaded
Hair lookup preview:
No lookup loaded

------------------------------------------------------------

Mexican data shape: No data loaded
Mexican lookup preview:
No lookup loaded


In [ ]:
# TEST 2: Apply a simple business-name filter on Hair data

# Make sure df_hair and lookup_hair exist from TEST 1
if df_hair is None or lookup_hair is None:
    print("Run Test 1 first so df_hair and lookup_hair are defined.")
else:
    sample_name = lookup_hair.select('name').to_series().to_list()[0]
    print("Sample business name:", sample_name)

    df_subset, label = apply_user_filters(
        df_full=df_hair,
        business_lookup=lookup_hair,
        business_id_filter=None,
        business_name=sample_name,
        state_filter=None,
        city_filter=None
    )

    print("Subset label:", label)
    print("Subset shape:", df_subset.shape)

Run Test 1 first so df_hair and lookup_hair are defined.


## Descriptive Analysis Tests

In [ ]:
#TEST 3: Line Charts for Descriptive Statistics
line_chart, pie_chart, best, worst = descriptive_analysis(df_hair, lookup_hair)
line_chart

AttributeError: 'NoneType' object has no attribute 'columns'

In [ ]:
#TEST #4: Top 5 Best & Top 5 Worst
line_chart, pie_chart, best, worst = descriptive_analysis(df_hair, lookup_hair)

print("Top 5 Best Locations:")
print(best)

print("\nTop 5 Worst Locations:")
print(worst)

AttributeError: 'NoneType' object has no attribute 'columns'

## Predictive Analysis Tests

In [ ]:
[c for c in df_hair.columns if "_pct" in c.lower()]

In [ ]:
[c for c in df_hair.columns if "emotion" in c.lower()]

In [ ]:
#TEST #5: Predictive Analysis Forecast
forecast_chart, predicted_star = predictive_analysis(df_hair, periods=6)
print("Predicted star after 6 months:", predicted_star)
forecast_chart